# 🚀 大模型客服助手项目 - 深入学习计划

本学习计划按照 **理论 → 简易实现 → 工程应用** 的顺序，带你深入理解这个项目的核心技术。

## 📋 学习路径总览

| 阶段 | 主题 | 难度 |
|------|------|------|
| 1 | Python异步编程 asyncio | ⭐⭐ |
| 2 | FastAPI与依赖注入 | ⭐⭐⭐ |
| 3 | LangChain核心概念 | ⭐⭐⭐ |
| 4 | RAG检索增强生成 | ⭐⭐⭐⭐ |
| 5 | 向量数据库Chroma | ⭐⭐⭐ |
| 6 | 多级缓存架构 | ⭐⭐⭐⭐ |
| 7 | NLP任务实现 | ⭐⭐⭐ |
| 8 | LLM微调技术 | ⭐⭐⭐⭐⭐ |
| 9 | 系统设计面试 | ⭐⭐⭐⭐⭐ |

---

# Stage 1: Python异步编程 asyncio

## 1.1 理论：asyncio到底是什么？

**底层原理：**

asyncio是Python的协程异步I/O框架，核心基于 **事件循环(Event Loop)** 和 **协程(Coroutine)** 。

```
┌─────────────────────────────────────────────────────┐
│                    事件循环                         │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐            │
│  │协程1    │  │协程2    │  │协程3    │  ...       │
│  │(等待IO) │  │(等待IO) │  │(执行中) │            │
│  └────┬────┘  └────┬────┘  └────┬────┘            │
│       │             │             │                 │
│  ←────┴─────────────┴─────────────┴────→           │
│              任务调度(单线程)                       │
└─────────────────────────────────────────────────────┘
```

**关键概念：**
- **协程(Coroutine)**：可暂停/恢复的函数，用`async def`定义
- **事件循环(Event Loop)**：调度协程执行的循环机制
- **Future/Task**：协程的包装器，代表未完成的异步操作
- **await**：暂停当前协程，等待另一个协程完成

**为什么快？**

当一个协程等待I/O（如API请求）时，事件循环会切换到其他可执行的协程，而不是阻塞等待。

## 1.2 简易实现

In [ ]:
# 第一个asyncio程序
import asyncio

async def say_hello(name: str, delay: int):
    """模拟异步操作"""
    print(f"[{name}] 开始执行...")
    await asyncio.sleep(delay)  # 模拟I/O等待
    print(f"[{name}] 完成!")
    return f"Hello {name}"

# 顺序执行（不好）
async def sequential():
    result1 = await say_hello("任务1", 1)
    result2 = await say_hello("任务2", 1)
    return result1, result2

# 并发执行（好！）
async def concurrent():
    # asyncio.gather: 并发执行多个协程
    results = await asyncio.gather(
        say_hello("任务A", 2),
        say_hello("任务B", 1),
        say_hello("任务C", 1.5)
    )
    return results

# 运行
async def main():
    print("=== 顺序执行 ===")
    import time
    start = time.time()
    await sequential()
    print(f"耗时: {time.time() - start:.2f}s\n")

    print("=== 并发执行 ===")
    start = time.time()
    await concurrent()
    print(f"耗时: {time.time() - start:.2f}s")

# asyncio.run(main())
print("注意: 顺序执行耗时约为各任务之和，并发执行耗时约为最长任务的时间")

## 1.3 项目中的实际应用

查看 `app/core/llm_client.py` 中的 `batch_generate` 方法：

### ❌ 有缺陷的版本（串行执行）

In [ ]:
# 缺陷代码：串行for循环
def batch_generate(self, prompts: List[str], ...) -> List[str]:
    results = []
    for i, prompt in enumerate(prompts):  # 一个一个顺序执行
        result = self.generate_text(prompt, ...)  # 每次都等待API返回
        results.append(result)
    return results

# 如果有10个prompt，每个1秒，共需10秒！

### ✅ 改进版本（并发执行）

In [ ]:
# 改进代码：使用asyncio.gather并发执行
import asyncio

class LLMClient:
    async def batch_generate_async(
        self,
        prompts: List[str],
        max_concurrency: int = 10
    ) -> List[str]:
        """
        批量生成（并发版本）
        
        Args:
            prompts: 提示词列表
            max_concurrency: 最大并发数（控制API速率限制）
        
        Returns:
            生成结果列表
        """
        semaphore = asyncio.Semaphore(max_concurrency)  # 限制并发数
        
        async def generate_with_semaphore(prompt: str, index: int):
            async with semaphore:
                return await self.async_generate_text(prompt)
        
        # 创建所有任务
        tasks = [generate_with_semaphore(p, i) for i, p in enumerate(prompts)]
        
        # 并发执行所有任务
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # 处理异常结果
        return [r if isinstance(r, str) else "" for r in results]

# 如果有10个prompt，每个1秒，并发执行只需约1-2秒（取决于max_concurrency）

## 1.4 Semaphore控制并发原理

In [ ]:
# Semaphore就像一个带计数的锁
# max_concurrency=3 表示同时最多有3个协程在执行

async def demo_semaphore():
    semaphore = asyncio.Semaphore(2)  # 最多2个并发
    
    async def task(name: str):
        async with semaphore:
            print(f"{name} 开始")
            await asyncio.sleep(1)
            print(f"{name} 结束")
    
    # 4个任务，但只有2个能同时执行
    await asyncio.gather(
        task("任务1"),
        task("任务2"),
        task("任务3"),
        task("任务4")
    )

# 预期输出:
# 任务1 开始
# 任务2 开始
# 任务1 结束
# 任务3 开始
# 任务2 结束
# 任务4 开始
# 任务3 结束
# 任务4 结束
# 总耗时约4秒（2+2）而不是8秒

## 1.5 面试问答

**Q1: asyncio为什么能提高I/O密集型任务的效率？**

**参考答案：**

asyncio通过事件循环和协程实现并发。当协程等待I/O（如网络请求）时，事件循环会自动切换到其他可执行的协程。相比于传统的多线程或同步阻塞：

1. **无线程切换开销**：协程切换是程序级别的，不需要操作系统调度
2. **避免GIL限制**：单线程内的协程切换不受Python全局解释器锁限制
3. **内存占用低**：每个协程约几KB，而每个线程约1MB

**Q2: 什么时候不适合用asyncio？**

**参考答案：**

1. **CPU密集型任务**：如图像处理、加密计算等，应该用`multiprocessing`
2. **有大量同步阻塞代码**：改造代价高
3. **需要真正的并行**：需要绑卡/科学计算等受GIL影响的场景

---

# Stage 2: FastAPI与依赖注入

## 2.1 理论：什么是依赖注入？

**核心思想**：把依赖的创建权从使用方分离出来，由框架统一管理。

```
┌─────────────────────────────────────────────────────┐
│                    FastAPI                         │
│                                                     │
│   ┌─────────────┐    ┌─────────────┐              │
│   │ 路由处理器   │───→│  依赖注入    │              │
│   └─────────────┘    └──────┬──────┘              │
│                             │                      │
│                    ┌────────▼────────┐            │
│                    │   可注入对象    │            │
│                    │  • 数据库连接    │            │
│                    │  • LLM客户端    │            │
│                    │  • 缓存服务      │            │
│                    └─────────────────┘            │
└─────────────────────────────────────────────────────┘
```

**解决的问题：**
1. 硬依赖：不用在每个函数里`new LLMClient()`
2. 测试困难：mock困难，依赖紧耦合
3. 单例管理：连接池复用

## 2.2 项目中的依赖注入

In [ ]:
# 模拟项目中的依赖注入模式
from functools import lru_cache
from typing import Optional

# ============ 单例模式的配置类 ============
class Settings:
    def __init__(self):
        self.dashscope_api_key = "your-api-key"
        self.default_model = "qwen-plus"
        self.default_temperature = 0.7

# 使用@lru_cache实现单例
@lru_cache(maxsize=1)
def get_settings() -> Settings:
    """
    获取配置单例
    
    @lru_cache 工作原理：
    - 第一次调用时执行函数并缓存结果
    - 后续调用直接返回缓存结果
    - maxsize=1 表示最多缓存1个实例
    """
    return Settings()

# ============ LLM客户端类 ============
class LLMClient:
    def __init__(self, api_key: str, model: str):
        self.api_key = api_key
        self.model = model
        print(f"创建LLMClient实例: {model}")

# ============ 依赖注入函数 ============
def get_llm_client() -> LLMClient:
    """
    获取LLM客户端单例
    
    这是依赖注入的核心函数
    - 第一次调用时创建实例
    - 后续调用返回同一实例
    - 这样所有API请求复用同一个连接池
    """
    settings = get_settings()
    return LLMClient(
        api_key=settings.dashscope_api_key,
        model=settings.default_model
    )

# 测试单例
client1 = get_llm_client()
client2 = get_llm_client()
print(f"是同一个实例: {client1 is client2}")
# 输出: 是同一个实例: True

## 2.3 FastAPI中的依赖注入

In [ ]:
# FastAPI依赖注入示例
from fastapi import FastAPI, Depends, HTTPException
from typing import Annotated

# 模拟数据库
class Database:
    def __init__(self):
        self.connected = True

# 依赖项：获取数据库连接
def get_db() -> Database:
    """每次请求创建新的数据库连接（示例）"""
    db = Database()
    try:
        yield db  # 返回给路由处理器
    finally:
        pass  # 清理资源

# 依赖项：获取当前用户
def get_current_user(token: str = Depends(lambda: "fake-token")):
    if token != "valid-token":
        raise HTTPException(status_code=401, detail="无效的token")
    return {"user_id": 123, "name": "张三"}

# 创建FastAPI应用
app = FastAPI()

# 使用依赖注入
@app.get("/profile")
async def get_profile(
    current_user: dict = Depends(get_current_user),  # 注入用户
    db: Database = Depends(get_db)  # 注入数据库
):
    return {
        "user": current_user,
        "db_connected": db.connected
    }

# 等价于手动写：
# @app.get("/profile")
# async def get_profile():
#     token = "fake-token"
#     if token != "valid-token":
#         raise HTTPException(status_code=401)
#     current_user = {"user_id": 123}
#     db = Database()
#     return {"user": current_user, "db_connected": db.connected}

## 2.4 项目中的@lru_cache单例模式

查看 `app/config/settings.py`：

```python
@lru_cache(maxsize=1)
def get_settings() -> Settings:
    return Settings()

settings = get_settings()  # 全局单例
```

**@lru_cache vs 单例模式的区别：**

| 特性 | @lru_cache | 单例模式(Singleton) |
|------|------------|-------------------|
| 线程安全 | ✅ 安全 | ✅ 安全 |
| 延迟加载 | ✅ 首次调用才创建 | ✅ 首次调用才创建 |
| 可测试性 | ❌ 难以mock | ✅ 容易mock |
| 清除缓存 | 可以手动clear | 需要额外实现 |

## 2.5 面试问答

**Q1: @lru_cache(maxsize=1) 和手动实现单例有什么区别？**

**参考答案：**

功能上类似，但@lru_cache是**装饰器模式**，自动处理缓存逻辑：

```python
# @lru_cache版本
@lru_cache(maxsize=1)
def get_instance():
    return ExpensiveObject()

# 手动版本
_instance = None
def get_instance():
    global _instance
    if _instance is None:
        _instance = ExpensiveObject()
    return _instance
```

@lru_cache的优势：内置线程安全、缓存管理。但缺点是难以测试（缓存难以清除）。

**Q2: FastAPI依赖注入的yield关键字有什么用？**

**参考答案：**

`yield`用于**依赖清理/资源释放**：

```python
def get_db() -> Database:
    db = Database()
    yield db  # 返回给路由
    # 路由完成后执行这里
    db.close()  # 关闭连接
```

这类似于Java的`try-with-resources`或Go的`defer`。

---

## 2.6 FastAPI核心概念

### 2.6.1 FastAPI vs Flask对比

| 特性 | FastAPI | Flask |
|------|---------|-------|
| 异步 | 原生支持async/await | 需Flask-Ayncioxt |
| 类型校验 | 自动基于Pydantic | 手动处理 |
| 自动文档 | Swagger/ReDoc内置 | 需flask-restful |
| 性能 | 接近NodeJS | 较慢 |
| 依赖注入 | 原生支持 | 需Bluprint/装饰器 |

### 2.6.2 项目中的FastAPI入口

查看 `app/main.py`：
```python
from fastapi import FastAPI

# 创建应用
app = FastAPI(
    title=settings.APP_NAME,
    version=settings.APP_VERSION,
    description="大模型客服助手API文档",
    lifespan=lifespan,  # 生命周期管理
)

# 注册路由
app.include_router(knowledge_base_router, prefix=API_PREFIX)
app.include_router(script_recommender_router, prefix=API_PREFIX)
app.include_router(quality_inspector_router, prefix=API_PREFIX)
```

### 2.6.3 路由参数处理
```python
from fastapi import Path, Query, Body, Header
from pydantic import BaseModel

# 路径参数
@app.get("/items/{item_id}")
async def get_item(item_id: int = Path(..., ge=1)):
    # item_id: int 自动类型转换
    # Path(..., ge=1) 表示必填且>=1
    return {"item_id": item_id}

# 查询参数
@app.get("/search")
async def search(q: str = Query(..., min_length=1), page: int = Query(1, ge=1)):
    # ?q=xxx&page=1
    return {"q": q, "page": page}

# 请求体
class Item(BaseModel):
    name: str
    price: float
    
@app.post("/items")
async def create_item(item: Item):
    return item

# Header参数
@app.get("/headers")
async def get_headers(token: str = Header(...)):
    return {"token": token}
```

### 2.6.4 响应模型
```python
from fastapi import Response
from typing import Optional

class ItemResponse(BaseModel):
    id: int
    name: str
    price: float
    tags: Optional[list[str]] = None

@app.get("/items/{item_id}", response_model=ItemResponse)
async def get_item(item_id: int):
    # FastAPI会自动：
    # 1. 验证返回数据
    # 2. 过滤敏感字段
    # 3. JSON序列化
    return {"id": item_id, "name": "test", "price": 10.0}
```

### 2.6.5 中间件
```python
from fastapi import FastAPI, Request
import time

app = FastAPI()

# 记录请求耗时的中间件
@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)
    return response
```

**中间件执行顺序（洋葱模型）**：
```
请求 → 中间件1进入 → 中间件2进入 → ... → 路由处理 → ... → 中间件2退出 → 中间件1退出 → 响应
```

### 2.6.6 项目路由注册详解

```python
# main.py中的路由注册
app.include_router(
    knowledge_base_router,  # 路由对象
    prefix=API_PREFIX,     # /api/v1
    tags=["知识库"]        # 文档分组
)

# knowledge_base_router定义 (app/api/routers/knowledge_base.py)
from fastapi import APIRouter

router = APIRouter()

@router.get("/knowledge/search")
async def search_knowledge(query: str):
    # 完整路径: GET /api/v1/knowledge/search
    return {"result": query}
```

### 2.6.7 异常处理
```python
from fastapi import HTTPException

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id not in items:
        raise HTTPException(
            status_code=404,
            detail=f"Item {item_id} not found",
            headers={"X-Error": "NotFound"}
        )
    return items[item_id]

# 全局异常处理（在main.py中）
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    return {"message": "Internal server error"}
```

### 2.6.8 lifespan生命周期
```python
from contextlib import asynccontextmanager
from fastapi import FastAPI

@asynccontextmanager
async def lifespan(app: FastAPI):
    # 启动时执行
    print("服务启动")
    await init_db()  # 初始化数据库
    await init_cache()  # 初始化缓存
    
    yield  # 请求处理阶段
    
    # 关闭时执行
    print("服务关闭")
    await close_db()  # 关闭连接

app = FastAPI(lifespan=lifespan)
```

### 2.6.9 面试问答

**Q1: FastAPI是如何实现自动参数类型校验的？**

**参考答案：**

FastAPI基于Pydantic的BaseModel实现自动校验。当你在函数参数中声明类型后，FastAPI会：

1. **拦截请求**：中间件捕获请求数据
2. **Pydantic验证**：调用BaseModel.parse_obj()验证数据
3. **自动转换**：如string→int转换
4. **错误处理**：验证失败返回422状态码

核心是Python的**类型提示(type hints)**和**装饰器**机制。

**Q2: 为什么FastAPI性能接近NodeJS/Go？**

**参考答案：**

1. **Starlette框架**：FastAPI基于Starlette，是目前最快的Python ASGI框架之一
2. **异步IO**：全程async/await，无阻塞IO
3. **最小化序列化**：Pydantic的序列化经过Cython优化
4. **类型提示优化**：支持mypy静态检查，提前发现类型问题

**Q3: FastAPI的中间件和依赖注入有什么区别？**

**参考答案：**

| | 中间件 | 依赖注入 |
|-------|--------|----------|
| 作用范围 | 所有请求 | 特定路由 |
| 执行顺序 | 洋葱模型，进出都执行 | 注入到函数参数 |
| 用途 | 日志、认证、全局限流 | 数据库连接、业务逻辑 |
| 获取结果 | 通过response对象 | 直接作为参数传入 |

---

# Stage 3: LangChain核心概念

## 3.1 LangChain是什么？

**官方定义**：LangChain是一个用于构建LLM应用的框架。

**核心组件**：

```
LangChain = LLMs + Prompts + Agents + Memory + Tools
```

| 组件 | 作用 | 项目中的例子 |
|------|------|-------------|
| **Models** | 大语言模型接口 | 通义千问(DashScope) |
| **Prompts** | 提示词模板管理 | PromptTemplates |
| **Chains** | 连接各组件的链 | RAG检索链 |
| **Agents** | 动态决定执行动作 | 意图分类Agent |
| **Memory** | 对话历史存储 | Redis缓存 |
| **Tools** | 外部能力调用 | 向量数据库检索 |

## 3.2 Prompt Templates（提示词模板）

In [1]:
# LangChain的PromptTemplate示例
from string import Template

# 项目中定义的模板（见constants.py）
INTENT_CLASSIFY_TEMPLATE = """
请对以下客户查询进行意图分类：

查询内容：${query}

可选类别：${categories}

请返回分类结果和置信度。
"""

def classify_intent(query: str, categories: list) -> str:
    """
    使用模板生成提示词
    
    为什么用模板而不是字符串拼接？
    1. 防止Prompt注入
    2. 结构清晰，便于维护
    3. 支持默认值和校验
    """
    template = Template(INTENT_CLASSIFY_TEMPLATE)
    return template.substitute(
        query=query,
        categories="、".join(categories)
    )

# 示例
result = classify_intent(
    query="我的贷款什么时候能批下来",
    categories=["账户查询", "业务办理", "投诉建议"]
)
print(result)


请对以下客户查询进行意图分类：

查询内容：我的贷款什么时候能批下来

可选类别：账户查询、业务办理、投诉建议

请返回分类结果和置信度。



## 3.3 Chain（链）的概念

Chain把多个组件串联起来工作：

```
用户输入 → PromptTemplate → LLM → 输出
     ↓
   Memory（记忆）
```

**RAG Chain 例子（项目中实际使用的）**：

```python
# 简化版的RAG链
class RAGChain:
    def __init__(self, llm_client, vector_db):
        self.llm = llm_client
        self.vector_db = vector_db
    
    def invoke(self, query: str) -> str:
        # Step 1: 检索相关文档
        docs = self.vector_db.query(query, n_results=3)
        
        # Step 2: 构建上下文
        context = "\n".join([doc["content"] for doc in docs])
        
        # Step 3: 生成回复
        prompt = f"基于以下信息回答：{context}\n\n问题：{query}"
        return self.llm.generate_text(prompt)
```

## 3.4 项目中LangChain的实际应用

In [ ]:
# 项目中的RAG实现（简化版）

class KnowledgeBaseAssistant:
    """
    知识库助手 - 使用RAG架构
    
    RAG = Retrieval Augmented Generation
    核心思想：先检索相关知识，再让LLM生成答案
    """
    
    def __init__(self, llm_client, vector_db):
        self.llm = llm_client
        self.vector_db = vector_db
    
    def answer(self, question: str) -> dict:
        # 1. 意图识别（Agent角色）
        intent = self.llm.classify_intent(
            text=question,
            categories=["账户查询", "业务办理", "技术支持"]
        )
        
        # 2. 检索相关知识
        retrieved_docs = self.vector_db.query_with_score(
            query_text=question,
            n_results=5
        )
        
        # 3. 构建Prompt（包含检索到的知识）
        context = "\n".join([
            f"- {doc['document']}" 
            for doc in retrieved_docs[:3]  # 取top3
        ])
        
        prompt = f"""
基于以下知识回答用户问题：

知识库：
${context}

用户问题：${question}

请给出专业、准确的回答。
"""
        
        # 4. LLM生成
        answer = self.llm.chat([
            {"role": "system", "content": prompt},
            {"role": "user", "content": question}
        ])
        
        return {
            "answer": answer,
            "intent": intent,
            "sources": retrieved_docs[:3]
        }

## 3.5 面试问答

**Q1: LangChain的Chain和Agent有什么区别？**

**参考答案：**

| | Chain | Agent |
|-------|-------|-------|
| 执行方式 | 预定义的静态序列 | 动态决定下一步 |
| 动作选择 | 固定的LLM→Output | LLM决定是否调用工具 |
| 适用场景 | 固定的RAG、SQL生成 | 多工具协同、复杂推理 |

Agent更像是一个**控制器**，能根据输入动态选择执行路径。

**Q2: RAG的核心思想是什么？有什么局限性？**

**参考答案：**

**核心思想**：检索+生成，先从知识库找到相关内容，再让LLM基于这些内容生成答案。

**优点**：
- 可更新知识（更新向量数据库即可）
- 减少幻觉（LLM回答基于真实检索内容）
- 可追溯（答案有据可查）

**局限性**：
- 检索质量决定生成质量
- 复杂推理（如多跳问题）效果差
- 上下文长度限制

---

# Stage 4: 向量数据库与RAG

## 4.1 理论：什么是向量数据库？

**核心概念**：

1. **Embedding（嵌入）**：把文本转换为向量
2. **相似度检索**：用向量距离衡量文本相似度

```
文本 → Embedding模型 → 向量 → 存储到向量数据库

查询 → Embedding → 向量 → 在数据库中找最近的向量 → 返回原始文本
```

**向量距离度量方式**：

| 方式 | 公式 | 适用场景 |
|------|------|---------|
| 欧氏距离 | √(Σ(a-b)²) | 图像/音频 |
| 余弦相似度 | cos(θ) | 文本（常用） |
| 点积 | a·b | 推荐系统 |

## 4.2 Chroma数据库项目应用

In [ ]:
# 项目中Chroma的使用（来自vector_db.py）

from typing import List, Dict, Any
import chromadb
from chromadb.config import Settings

class VectorDBClient:
    """
    Chroma向量数据库客户端
    
    Chroma特点：
    1. 轻量级，易于部署
    2. 支持多种embedding后端
    3. 有服务器模式和嵌入模式
    """
    
    def __init__(self, collection_name: str = "customer_service_knowledge"):
        self.collection_name = collection_name
        
        # 初始化Chroma客户端
        # 有两种模式：
        # 1. 服务器模式：chroma_server_host + chroma_server_http_port
        # 2. 持久化模式：persist_directory
        self._client = chromadb.Client(Settings(
            chroma_server_host="localhost",
            chroma_server_http_port=8000
        ))
        
        # 获取或创建集合（类似表）
        self._collection = self._client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "客服知识库向量数据"}
        )
    
    def add_documents(
        self,
        documents: List[str],
        metadatas: List[Dict[str, Any]] = None,
        ids: List[str] = None
    ) -> List[str]:
        """
        添加文档到向量数据库
        
        参数说明：
        - documents: 文档内容列表
        - metadatas: 元数据（如来源、时间等）
        - ids: 文档ID（可选，自动生成UUID）
        """
        if ids is None:
            import uuid
            ids = [str(uuid.uuid4()) for _ in documents]
        
        if metadatas is None:
            metadatas = [{} for _ in documents]
        
        self._collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        
        return ids
    
    def query(
        self,
        query_texts: List[str],
        n_results: int = 5,
        where: Dict[str, Any] = None
    ) -> Dict[str, Any]:
        """
        查询相似文档
        
        参数说明：
        - query_texts: 查询文本
        - n_results: 返回结果数量
        - where: 过滤条件（如只查询特定类型的文档）
        
        返回：
        - documents: 匹配的文档列表
        - distances: 向量距离（越小越相似）
        - metadatas: 元数据
        - ids: 文档ID
        """
        return self._collection.query(
            query_texts=query_texts,
            n_results=n_results,
            where=where
        )

## 4.3 向量数据库的三层降级策略（Bug修复）

### ❌ 有缺陷的版本

In [ ]:
# 缺陷代码：没有降级策略，Chroma挂了就没了
class BadVectorDBClient:
    def __init__(self):
        self._client = chromadb.Client(Settings(
            chroma_server_host=settings.CHROMA_HOST,
            chroma_server_http_port=settings.CHROMA_PORT
        ))
        # 如果连接失败，直接抛异常，没有任何降级
        self._collection = self._client.get_or_create_collection(...)
    
    def query(self, query_texts, n_results=5):
        # 如果Chroma服务器挂了，这里就会失败
        return self._collection.query(query_texts, n_results=n_results)

# 问题：生产环境中Chroma可能不可用，导致服务中断

### ✅ 改进版本：三层降级策略

In [ ]:
from enum import Enum

class VectorDBMode(Enum):
    """向量数据库模式"""
    SERVER = "server"        # 服务模式：连接远程Chroma服务器
    PERSISTENT = "persistent" # 持久化模式：本地Chroma
    EPHEMERAL = "ephemeral"  # 临时模式：纯内存（重启丢失）

class ResilientVectorDBClient:
    """
    具有三层降级策略的向量数据库客户端
    
    降级顺序：Server → Persistent → Ephemeral
    """
    
    def __init__(self, collection_name: str):
        self.collection_name = collection_name
        self._client = None
        self._collection = None
        self._current_mode = None
        self._init_client()
    
    def _init_client(self):
        """初始化客户端，失败则降级"""
        
        # Tier 1: 尝试服务器模式
        try:
            import chromadb
            from chromadb.config import Settings
            
            self._client = chromadb.Client(Settings(
                chroma_server_host=settings.CHROMA_HOST,
                chroma_server_http_port=settings.CHROMA_PORT
            ))
            self._collection = self._client.get_or_create_collection(
                name=self.collection_name
            )
            self._current_mode = VectorDBMode.SERVER
            print(f"✅ 向量数据库使用服务器模式")
            return
        except Exception as e:
            print(f"⚠️ 服务器模式失败: {e}")
        
        # Tier 2: 尝试持久化模式
        try:
            self._client = chromadb.Client(Settings(
                persist_directory="./chroma_data"
            ))
            self._collection = self._client.get_or_create_collection(
                name=self.collection_name
            )
            self._current_mode = VectorDBMode.PERSISTENT
            print(f"✅ 向量数据库使用持久化模式")
            return
        except Exception as e:
            print(f"⚠️ 持久化模式失败: {e}")
        
        # Tier 3: 最终降级到内存模式
        try:
            import chromadb
            from chromadb.config import Settings
            
            # ClientAPI默认就是ephemeral模式
            self._client = chromadb.EphemeralClient()
            self._collection = self._client.create_collection(
                name=self.collection_name
            )
            self._current_mode = VectorDBMode.EPHEMERAL
            print(f"✅ 向量数据库使用内存模式（警告：重启后数据丢失）")
            return
        except Exception as e:
            raise Exception(f"所有向量数据库模式都失败了: {e}")
    
    def get_mode(self) -> VectorDBMode:
        """获取当前模式"""
        return self._current_mode

## 4.4 面试问答

**Q1: Chroma和Milvus/Pinecone有什么区别？**

**参考答案：**

| 特性 | Chroma | Milvus | Pinecone |
|------|--------|--------|----------|
| 部署 | 轻量/嵌入 | K8s/分布式 | 云服务 |
| 规模 | 百万级 | 十亿级 | 十亿级 |
| 延迟 | 低 | 中 | 低 |
| 成本 | 免费 | 开源 | 付费 |
| 适用 | 原型/中小规模 | 生产级 | 云原生 |

**Q2: 如何优化向量检索的召回率？**

**参考答案：**

1. **Embedding优化**：选择更好的embedding模型（如BGE、OpenAI-embedding-3）
2. **分块策略**：合理切割文档（语义完整性、长度控制）
3. **混合检索**：结合关键词搜索（BM25）和向量搜索
4. **重排序**：用更精确的模型（如Cross-Encoder）对初筛结果重排
5. **索引优化**：HNSW/IVF等索引参数调优

---

# Stage 5: 多级缓存架构

## 5.1 理论：为什么需要多级缓存？

**问题背景**：

```
用户请求 → LLM API → 响应慢(1-3秒) → 成本高
                    ↑
               大量重复请求
```

**解决思路**：缓存+多级+穿透保护

```
┌──────────────────────────────────────────────────┐
│                   请求                            │
│                     ↓                           │
│  ┌─────────────────────────────────────────┐    │
│  │        L1: 内存缓存 (Caffeine/本地)      │    │
│  │        延迟: <1ms  容量: ~10000          │    │
│  └─────────────────┬───────────────────────┘    │
│                    ↓ 未命中                      │
│  ┌─────────────────────────────────────────┐    │
│  │        L2: Redis缓存                    │    │
│  │        延迟: 1-5ms  容量: ~1000000      │    │
│  └─────────────────┬───────────────────────┘    │
│                    ↓ 未命中                      │
│  ┌─────────────────────────────────────────┐    │
│  │        L3: 向量数据库检索              │    │
│  │        延迟: 10-50ms                   │    │
│  └─────────────────┬───────────────────────┘    │
│                    ↓ 未命中                      │
│  ┌─────────────────────────────────────────┐    │
│  │        LLM API (最慢/最贵)              │    │
│  └─────────────────────────────────────────┘    │
└──────────────────────────────────────────────────┘
```

## 5.2 Bloom Filter（布隆过滤器）

In [ ]:
# Bloom Filter 实现缓存穿透保护
import hashlib
from typing import List

class BloomFilter:
    """
    布隆过滤器 - 用于判断元素"一定不存在"还是"可能存在"
    
    工作原理：
    1. 使用多个哈希函数映射到位数组
    2. 查询时，所有哈希位置都为1才认为可能存在
    
    在缓存场景的应用：
    - 如果BloomFilter说"不存在"，直接返回缓存未命中
    - 如果说"可能存在"，再去查缓存/数据库
    - 防止大量不存在的数据穿透到数据库
    """
    
    def __init__(self, size: int = 1000000, hash_count: int = 7):
        self.size = size
        self.hash_count = hash_count
        self.bit_array = [0] * size
    
    def _hashes(self, item: str) -> List[int]:
        """生成多个哈希值"""
        result = []
        for i in range(self.hash_count):
            hash_input = f"{item}_{i}".encode()
            hash_value = int(hashlib.md5(hash_input).hexdigest(), 16)
            result.append(hash_value % self.size)
        return result
    
    def add(self, item: str):
        """添加元素"""
        for pos in self._hashes(item):
            self.bit_array[pos] = 1
    
    def might_contain(self, item: str) -> bool:
        """检查元素是否可能存在"""
        return all(self.bit_array[pos] == 1 for pos in self._hashes(item))

# 示例
bf = BloomFilter()

# 添加一些存在的key
bf.add("产品ID:123")
bf.add("产品ID:456")

# 查询
print(bf.might_contain("产品ID:123"))  # True - 一定存在
print(bf.might_contain("产品ID:999"))  # False - 一定不存在

# 注意：可能有假阳性（不存在但说可能存在）
# 但不会有假阴性（存在说一定不存在）

# 这正是我们需要的特性：
# - 说"不存在" → 真的不存在，不用查缓存
# - 说"可能存在" → 可能存在，去查缓存

## 5.3 项目中的缓存策略

In [ ]:
# 项目中的多级缓存实现

class CacheStrategy:
    """
    多级缓存策略
    
    L1: 本地内存缓存 - 响应最快
    L2: Redis分布式缓存 - 跨服务共享
    L3: LLM API - 最终来源
    """
    
    def __init__(self, redis_client, llm_client):
        self.redis = redis_client
        self.llm = llm_client
        self.local_cache = {}  # L1: 简单的dict做本地缓存
        self.bloom_filter = BloomFilter()  # 穿透保护
    
    def get_response(self, prompt: str) -> str:
        """
        获取响应的完整流程
        
        1. BloomFilter检查：是否可能存在
        2. L1本地缓存查询
        3. L2 Redis缓存查询
        4. LLM API调用
        """
        cache_key = self._make_cache_key(prompt)
        
        # Step 1: BloomFilter检查
        if not self.bloom_filter.might_contain(cache_key):
            # 一定不存在，直接返回空（或者调用LLM生成后缓存）
            pass
        
        # Step 2: L1本地缓存
        if cache_key in self.local_cache:
            print("L1缓存命中")
            return self.local_cache[cache_key]
        
        # Step 3: L2 Redis缓存
        cached = self.redis.get(cache_key)
        if cached:
            print("L2缓存命中")
            # 回填L1
            self.local_cache[cache_key] = cached
            return cached
        
        # Step 4: 调用LLM API
        print("缓存未命中，调用LLM")
        response = self.llm.generate_text(prompt)
        
        # 回填缓存
        self.local_cache[cache_key] = response
        self.redis.setex(cache_key, 3600, response)  # 1小时过期
        self.bloom_filter.add(cache_key)
        
        return response
    
    def _make_cache_key(self, prompt: str) -> str:
        """生成缓存key"""
        import hashlib
        return hashlib.md5(prompt.encode()).hexdigest()

## 5.4 面试问答

**Q1: 缓存穿透、击穿、雪崩有什么区别？如何解决？**

**参考答案：**

| 问题 | 描述 | 解决方案 |
|------|------|---------|
| **穿透** | 查询不存在的数据，穿透到DB | BloomFilter过滤、缓存空值 |
| **击穿** | 热点key过期，瞬间大量请求打DB | 互斥锁、永不过期+异步更新 |
| **雪崩** | 大量key同时过期 | 随机TTL、Redis持久化 |

**Q2: Redis和本地缓存如何选择？**

**参考答案：**

| 场景 | 推荐 | 原因 |
|------|------|------|
| 分布式环境 | Redis | 本地缓存不共享 |
| 超低延迟 | 本地缓存 | 节省一次网络往返 |
| 数据一致性 | Redis | 更新可以立即同步 |
| 大容量 | Redis | 本地内存有限 |

一般组合使用：本地做L1，Redis做L2。

---

# Stage 6: NLP任务实现

## 6.1 项目中的NLP任务

本项目涉及以下NLP任务：

| 任务 | 说明 | 技术方案 |
|------|------|---------|
| **意图识别** | 判断用户想干什么 | LLM分类 |
| **实体识别(NER)** | 提取关键实体 | LLM抽取 |
| **情感分类** | 判断客户情绪 | LLM分类 |
| **对话摘要** | 压缩长对话 | LLM生成 |

## 6.2 意图识别实现

In [ ]:
# 意图识别 - 项目中的实现

class IntentClassifier:
    """
    基于LLM的意图分类器
    
    工作原理：
    1. 构建分类Prompt
    2. 调用LLM生成
    3. 解析结果
    """
    
    # 预定义的意图类别
    CATEGORIES = [
        "账户查询",      # 查询账户余额、流水等
        "业务办理",      # 办理新业务、变更套餐等
        "投诉建议",      # 投诉服务、提出建议等
        "技术支持",      # 技术问题求助
        "其他",          # 无法分类
    ]
    
    def __init__(self, llm_client):
        self.llm = llm_client
    
    def classify(self, query: str) -> dict:
        """
        对用户查询进行意图分类
        
        Args:
            query: 用户输入的查询
        
        Returns:
            dict: 包含intent和confidence
        """
        # 构建Prompt
        prompt = f"""
请对以下客户查询进行意图分类：

查询内容：{query}

可选类别：{", ".join(self.CATEGORIES)}

请只返回一个最可能的类别，不要解释。
"""
        
        # 调用LLM
        response = self.llm.generate_text(prompt)
        
        # 解析结果
        intent = self._parse_response(response)
        
        return {
            "intent": intent,
            "confidence": 0.85,  # 简化的置信度
            "query": query
        }
    
    def _parse_response(self, response: str) -> str:
        """解析LLM返回结果"""
        # 简单匹配：看返回文本中包含哪个类别
        for category in self.CATEGORIES:
            if category in response:
                return category
        return "其他"

# 使用示例
# classifier = IntentClassifier(llm_client)
# result = classifier.classify("我的贷款什么时候能批下来")
# print(result)  # {'intent': '账户查询', 'confidence': 0.85, 'query': '我的贷款什么时候能批下来'}

## 6.3 情感分析实现

In [ ]:
# 情感分析 - 项目中的实现

class SentimentAnalyzer:
    """
    基于LLM的情感分析器
    
    情感类别：积极、中立、消极、愤怒、焦虑
    """
    
    EMOTIONS = ["积极", "中立", "消极", "愤怒", "焦虑"]
    
    def __init__(self, llm_client):
        self.llm = llm_client
    
    def analyze(self, text: str) -> dict:
        """
        分析文本的情感
        
        Args:
            text: 待分析的文本
        
        Returns:
            dict: 包含emotion和intensity (0-1)
        """
        prompt = f"""
请分析以下客服对话中客户的情感状态：

对话内容：{text}

情感类别：{", ".join(self.EMOTIONS)}

请返回情感类别和强度（0-1之间的数字），格式如下：
情感：[类别]
强度：[0-1的数字]
"""
        
        response = self.llm.generate_text(prompt)
        return self._parse_response(response)
    
    def _parse_response(self, response: str) -> dict:
        """解析LLM返回"""
        result = {"emotion": "中立", "intensity": 0.5}
        
        for emotion in self.EMOTIONS:
            if emotion in response:
                result["emotion"] = emotion
                break
        
        # 简单提取数字
        import re
        numbers = re.findall(r'0?\.\d+', response)
        if numbers:
            result["intensity"] = float(numbers[0])
        
        return result

# 使用示例
# analyzer = SentimentAnalyzer(llm_client)
# result = analyzer.analyze("你们这个产品太烂了，等了这么久还没处理好！")
# print(result)  # {'emotion': '愤怒', 'intensity': 0.9}

## 6.4 面试问答

**Q1: 传统NLP和LLM-based NLP有什么区别？**

**参考答案：**

| 方面 | 传统NLP | LLM-based NLP |
|------|---------|----------------|
| 模型 | BERT/RoBERTa等 | GPT/通义千问等 |
| 训练 | 需标注数据微调 | 预训练+Prompt |
| 泛化 | 领域迁移需微调 | Zero-shot/Few-shot |
| 成本 | 低（可本地部署） | 高（需API调用） |
| 延迟 | 低（本地模型） | 高（网络请求） |

**Q2: LLM做分类任务的Prompt怎么设计？**

**参考答案：**

关键要素：
1. **任务说明**：明确告诉LLM要做什么
2. **类别定义**：列出所有可能的类别
3. **输出格式**：指定返回格式（如JSON）
4. **few-shot示例**（可选）：给1-3个示例

```python
prompt = """
任务：判断用户查询属于哪个意图类别

类别：
1. 账户查询 - 查询余额、流水、账单等
2. 业务办理 - 开通服务、变更套餐等
3. 投诉建议 - 投诉服务、提出建议等

用户输入：{user_query}

请返回JSON格式：{"intent": "类别名称"}
"""
```

---

# Stage 7: LLM微调技术

## 7.1 理论：LLM训练的几个阶段

```
预训练(Pretrain) → 指令微调(SFT) → 人类反馈对齐(RLHF)
     ↓                  ↓                  ↓
 海量无标注文本     人工标注的问答对     人类偏好数据
 学习通用知识      学习任务能力        学习人类偏好
```

| 阶段 | 目的 | 数据 | 方法 |
|------|------|------|------|
| **Pretrain** | 学习语言模型 | 数十亿文本 | 自回归训练 |
| **SFT** | 学习任务能力 | 万级问答对 | 有监督学习 |
| **RLHF** | 对齐人类偏好 | 人类排序数据 | PPO算法 |
| **LoRA** | 高效微调 | 可少至百条 | 低秩矩阵分解 |

## 7.2 LoRA详解

In [ ]:
# LoRA (Low-Rank Adaptation) 原理

class LoRAExplanation:
    """
    LoRA的核心思想：
    
    原始全微调：更新W，W' = W + ΔW
    ΔW 是 m×n 的矩阵，参数量 = m×n
    
    LoRA：ΔW = B×A，其中B是m×r，A是r×n
    参数量 = m×r + r×n
    
    由于 r << min(m, n)，参数量大大减少
    
    例如：
    - m=4096, n=4096 的全量微调：16M参数
    - r=8 的LoRA：4096×8 + 8×4096 = 65K参数
    - 压缩比：250倍！
    """
    
    @staticmethod
    def illustration():
        print("="*60)
        print("全量微调 vs LoRA")
        print("="*60)
        print("""
全量微调:
        W (m×n)                ΔW (m×n)            W' = W + ΔW
    ┌─────────┐          ┌─────────┐         ┌─────────┐
    │         │    +     │         │    =    │         │
    │  4096×4096  │        │  4096×4096  │        │  4096×4096  │
    │ 16M参数  │          │ 16M参数  │         │ 32M参数  │
    └─────────┘          └─────────┘         └─────────┘

LoRA (r=8):
        W (m×n)                ΔW = B×A           W' = W + B×A
    ┌─────────┐          ┌───────┐ ┌───────┐  ┌─────────┐
    │         │    +     │4096×8 │×│8×4096 │  │         │
    │  4096×4096  │          └───────┘ └───────┘  │  4096×4096  │
    │  固定冻结  │          └───────┘           │ + 65K参数  │
    └─────────┘                                 └─────────┘
                                                     ↑
                                          只有B和A需要训练
        """)

LoRAExplanation.illustration()

## 7.3 项目中是否用到微调？

**回答**：本项目**直接使用API调用通义千问**，没有在项目中进行模型微调。

但了解微调对于以下场景很重要：

1. **垂类领域优化**：通用LLM在金融、医疗等专业领域效果差，需要微调
2. **特定任务增强**：如客服话术生成、法律文书撰写等
3. **成本优化**：微调后可以用小模型达到大模型效果

**什么时候需要微调 vs RAG？**

| 场景 | 推荐方案 | 原因 |
|------|---------|------|
| 知识库问答 | RAG | 知识可更新 |
| 特定格式输出 | RAG+Prompt | 无需训练 |
| 特定语气/风格 | LoRA微调 | 需要内化风格 |
| 复杂推理模式 | LoRA微调 | 需要学习模式 |
| 实时知识 | RAG | 微调无法快速更新 |

## 7.4 面试问答

**Q1: 什么是RLHF？它的三个步骤是什么？**

**参考答案：**

RLHF = Reinforcement Learning from Human Feedback

**三个步骤**：

1. **SFT（有监督微调）**：用人工标注的问答数据微调模型

2. **奖励模型训练**：让人类对模型输出排序，训练一个奖励模型RM
   - 输入：(prompt, response)
   - 输出：评分

3. **PPO强化学习**：用奖励模型作为信号，通过PPO算法优化策略
   - 目标：最大化奖励
   - 约束：KL散度限制，不让模型偏离SFT太远

**Q2: LoRA的原理？为什么能省参数？**

**参考答案：**

核心假设：预训练大模型的参数矩阵是低秩的。

全量微调时：W' = W + ΔW，ΔW是m×n矩阵

LoRA假设ΔW = BA，其中B是m×r，A是r×n

参数量从m×n变成m×r + r×n

由于r远小于m和n（如r=8），参数量大幅减少。

训练时只更新A和B，推理时把BA加到W上，等于原模型+W'。

---

# Stage 8: 系统设计面试题

## 8.1 设计一个客服对话系统

**问题**：如何设计一个日活100万的智能客服系统？

**解答框架**：

### 1. 核心指标计算

```python
# 估算QPS
DAU = 1_000_000  # 日活
avg_requests_per_user = 10  # 每人每天10次
daily_requests = DAU * 10 = 10_000_000  # 1千万
QPS = daily_requests / 86400 ≈ 116  # 平均116

# 峰值QPS（假设集中在8小时）
peak_factor = 5  # 峰值是平均的5倍
peak_QPS = 116 * 5 = 580

# LLM API延迟假设：500ms
# 需要并发连接数：580 * 0.5 = 290
```

### 2. 整体架构

In [ ]:
# 客服系统架构设计

architecture = """
┌─────────────────────────────────────────────────────────────────┐
│                         用户请求                                │
│                     (APP/网页/电话)                              │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                      负载均衡层                                   │
│                   (Nginx/K8s Ingress)                           │
└──────────────────────────┬──────────────────────────────────────┘
                           │
         ┌─────────────────┼─────────────────┐
         │                 │                 │
         ▼                 ▼                 ▼
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  FastAPI 1  │    │  FastAPI 2  │    │  FastAPI N  │
│  (副本)     │    │  (副本)     │    │  (副本)     │
└──────┬──────┘    └──────┬──────┘    └──────┬──────┘
       │                   │                   │
       └───────────────────┼───────────────────┘
                           │
         ┌─────────────────┼─────────────────┐
         │                 │                 │
         ▼                 ▼                 ▼
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  Redis L1   │    │  Redis L2   │    │   Chroma    │
│  (本地缓存) │    │ (分布式缓存)│    │ (向量检索)  │
└─────────────┘    └──────┬──────┘    └─────────────┘
                           │
                           ▼
              ┌─────────────────────────┐
              │      LLM API            │
              │   (通义千问/ChatGPT)     │
              └─────────────────────────┘
"""
print(architecture)

### 3. 关键组件设计

```
┌─────────────────────────────────────────────────────────────────┐
│                      对话管理服务                               │
│  • 会话状态管理（Context Window）                               │
│  • 多轮对话状态存储（Redis Hash）                              │
│  • 超时处理（沉默超时自动结束）                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                      意图识别服务                               │
│  • 轻量级分类模型（或LLM）                                      │
│  • 多级意图树                                                   │
│  • 置信度阈值（低于阈值转人工）                                  │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                      RAG检索服务                                 │
│  • 知识库向量化和索引                                           │
│  • 混合检索（向量+关键词）                                      │
│  • 重排序（Cross-Encoder）                                      │
└─────────────────────────────────────────────────────────────────┘
```

### 4. 高可用设计

| 策略 | 实现 | 作用 |
|------|------|------|
| **降级** | LLM API不可用时返回预设回复 | 防止级联失败 |
| **限流** | 令牌桶算法限制QPS | 保护后端 |
| **重试** | 指数退避重试（3次）| 处理瞬时故障 |
| **熔断** | 连续失败触发熔断 | 快速失败 |
| **幂等** | 对话ID去重 | 防止重复处理 |

## 8.2 面试问答

**Q1: 如何解决LLM的幻觉问题？**

**参考答案：**

1. **RAG**：让LLM回答基于检索到的真实知识
2. **Prompt约束**：明确要求"不知道就说不知道"
3. **知识蒸馏**：用更小的模型蒸馏大模型的知识
4. **Chain of Thought**：让模型一步步推理，减少瞎猜
5. **结果验证**：用另一个模型验证回答的准确性

**Q2: 如何降低LLM API的成本？**

**参考答案：**

1. **缓存**：相同问题直接返回缓存结果
2. **摘要**：对长对话进行摘要，减少token
3. **小模型**：简单问题用小模型，复杂问题用大模型
4. **批量处理**：积攒请求批量调用
5. **参数调优**：适当降低max_tokens，避免浪费

**Q3: 如何保证对话的隐私安全？**

**参考答案：**

1. **数据脱敏**：对话前去除敏感信息（手机号、身份证）
2. **加密传输**：HTTPS/TLS加密
3. **数据隔离**：不同租户数据隔离存储
4. **日志审计**：记录操作日志，追溯泄密
5. **合规存储**：按照GDPR等法规存储数据

---

# 📚 附录：面试高频问题速记

## Python基础

| 问题 | 答案要点 |
|------|----------|
| @lru_cache原理 | 哈希表缓存，maxsize控制大小，thread-safe |
| asyncio vs threading | 单线程协程 vs 多线程，适用于I/O密集型 |
| 生成器 vs 迭代器 | 生成器是yield的函数，迭代器是实现了__next__的对象 |

## FastAPI

| 问题 | 答案要点 |
|------|----------|
| 依赖注入优势 | 解耦、易测试、复用、生命周期管理 |
| 中间件顺序 | 按添加顺序执行（洋葱模型） |
| lifespan上下文 | 启动/关闭时执行清理资源 |

## LLM相关

| 问题 | 答案要点 |
|------|----------|
| RAG核心 | 检索+生成，先查知识库再让LLM回答 |
| 向量数据库作用 | 存储embedding，实现语义检索 |
| LoRA原理 | 低秩矩阵分解，大幅减少微调参数量 |
| RLHF三步 | SFT→奖励模型→PPO强化学习 |

## 系统设计

| 问题 | 答案要点 |
|------|----------|
| 缓存穿透 | BloomFilter + 缓存空值 |
| 缓存击穿 | 互斥锁 + 永不过期 + 异步更新 |
| 缓存雪崩 | 随机TTL + Redis持久化 |
| 高可用降级 | 服务分级，核心功能优先 |

---

# 🎯 学习建议

1. **循序渐进**：先理解asyncio和依赖注入，再学LangChain
2. **动手实践**：每个章节的代码都要实际运行一遍
3. **结合项目**：把学到的东西对照项目代码找对应实现
4. **面试复盘**：每学完一个模块，自己提几个面试问题

## 推荐阅读

- [LangChain官方文档](https://docs.langchain.com/)
- [FastAPI依赖注入](https://fastapi.tiangolo.com/tutorial/dependencies/)
- [Chroma向量数据库](https://docs.trychroma.com/)
- [LoRA原论文](https://arxiv.org/abs/2106.09685)
- [RLHF Explained](https://huggingface.co/blog/rlhf)

---

In [ ]:
print("✅ 学习计划完成！祝你在面试中取得好成绩！")